# Course 8: Privacy-Preserving Machine Learning
## Session 4: Auditing privacy

**Goal:** plant a canary, attack many DP training runs, and compare the measured lower bound on epsilon with the proven upper bound.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. The idea
The accountant gives an upper bound on epsilon. An audit gives a lower bound: train many times with and without one planted example (the canary), try to tell which is which, and convert the attacker's success into a statement about epsilon.

If a mechanism is (epsilon, delta)-DP, any attacker satisfies `TPR <= e^epsilon * FPR + delta`. So a measured (TPR, FPR) pair gives `epsilon >= ln((TPR - delta) / FPR)`.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from math import comb, exp, log, sqrt
from torch.func import functional_call, vmap, grad

# ---- privacy accounting (Renyi DP of the sampled Gaussian mechanism, whole-number orders) ----
def rdp(q, sigma, alpha):
    """Renyi DP at order alpha of one DP-SGD step. q = sampling rate, sigma = noise multiplier."""
    if q == 1.0: return alpha / (2 * sigma ** 2)
    terms = [log(comb(alpha, k)) + (alpha - k) * log(1 - q) + k * log(q) + (k * k - k) / (2 * sigma ** 2) for k in range(alpha + 1)]
    m = max(terms)
    return (m + log(sum(exp(t - m) for t in terms))) / (alpha - 1)

def epsilon(q, sigma, steps, delta):
    """Convert composed Renyi DP to (epsilon, delta)-DP and keep the best order."""
    return min(steps * rdp(q, sigma, a) + log(1 / delta) / (a - 1) for a in range(2, 129))

# ---- DP-SGD ----
def dp_sgd(model, X, y, sigma, clip=1.0, lr=0.5, steps=300, q=0.1, seed=0, loss_fn=F.cross_entropy):
    """Poisson sampling, per-example clipping, Gaussian noise. sigma=0 with clip=inf is ordinary SGD."""
    g = torch.Generator().manual_seed(seed)
    params = {k: v.detach().clone() for k, v in model.named_parameters()}
    one = lambda p, x, t: loss_fn(functional_call(model, p, (x[None],)), t[None])
    per_example_grads = vmap(grad(one), in_dims=(None, 0, 0))
    n = len(X)
    for _ in range(steps):
        idx = torch.nonzero(torch.rand(n, generator=g) < q).squeeze(-1)
        grads = per_example_grads(params, X[idx], y[idx])
        norms = torch.sqrt(sum((v.flatten(1) ** 2).sum(1) for v in grads.values()))
        scale = (clip / (norms + 1e-12)).clamp(max=1.0)                     # shrink any gradient longer than clip
        for k in params:
            total = (grads[k] * scale.view(-1, *[1] * (grads[k].dim() - 1))).sum(0)
            if sigma > 0: total = total + torch.randn(total.shape, generator=g) * sigma * clip
            params[k] -= lr * total / (q * n)
    model.load_state_dict(params)
    return model

def mlp(d_in, n_out, hidden=32):
    return nn.Sequential(nn.Linear(d_in, hidden), nn.Tanh(), nn.Linear(hidden, n_out))

def accuracy(model, X, y):
    with torch.no_grad(): return (model(X).argmax(1) == y).float().mean().item()

In [ ]:
from sklearn.datasets import load_digits
from scipy.stats import beta
d = load_digits()
X = np.c_[d.data[:200] / 16.0, np.ones(200)]             # 200 examples, with a bias column
Y = np.eye(10)[d.target[:200]]
canary_x = np.zeros(65); canary_x[0] = 10.0               # pixel 0 is always blank in real digits, so the canary stands alone
canary_y = np.eye(10)[0]

def noisy_gd(include_canary, sigma, rng, steps=10, clip=1.0, lr=20.0):
    """Full-batch DP gradient descent on softmax regression. Returns the weight the canary pushes on."""
    Xs, Ys = (np.vstack([X, canary_x]), np.vstack([Y, canary_y])) if include_canary else (X, Y)
    W = np.zeros((65, 10))
    for _ in range(steps):
        Z = Xs @ W; P = np.exp(Z - Z.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
        R = P - Ys                                                        # per-example gradient is outer(x_i, R_i)
        norms = np.linalg.norm(Xs, axis=1) * np.linalg.norm(R, axis=1)
        R = R * np.minimum(1, clip / (norms + 1e-12))[:, None]
        G = Xs.T @ R + sigma * clip * rng.normal(size=W.shape)
        W -= lr * G / len(X)
    return W[0, 0]                                                        # white-box attacker reads this weight

### 2. Run the game 2,000 times

In [ ]:
def audit(sigma, trials=2000, delta=1e-5, seed=0):
    rng = np.random.default_rng(seed)
    b = rng.random(trials) < 0.5
    score = np.array([noisy_gd(bi, sigma, rng) for bi in b])
    half = trials // 2
    thr_grid = np.quantile(score[:half], np.linspace(0.02, 0.98, 49))
    # choose the threshold on the first half, then measure on the second half with 95% confidence bounds
    best, best_eps = None, -1
    for thr in thr_grid:
        tp, fp = ((score[:half] > thr) & b[:half]).sum(), ((score[:half] > thr) & ~b[:half]).sum()
        e = log(max(tp, 1) / max(fp, 1))
        if e > best_eps: best, best_eps = thr, e
    guess = score[half:] > best; bb = b[half:]
    tp, fp, P, N = (guess & bb).sum(), (guess & ~bb).sum(), bb.sum(), (~bb).sum()
    tpr_lo = beta.ppf(0.025, tp, P - tp + 1) if tp > 0 else 0.0           # Clopper-Pearson bounds
    fpr_hi = beta.ppf(0.975, fp + 1, N - fp) if fp < N else 1.0
    eps_lower = log(max(tpr_lo - delta, 1e-12) / fpr_hi)
    return tp / P, fp / N, max(eps_lower, 0.0)

STEPS, DELTA = 10, 1e-5
for sigma in [4.0, 2.0, 1.0, 0.5]:
    tpr, fpr, lower = audit(sigma)
    upper = epsilon(1.0, sigma, STEPS, DELTA)
    print(f"sigma {sigma:3.1f}   TPR {tpr:.3f}  FPR {fpr:.3f}   measured lower bound {lower:5.2f}   proven upper bound {upper:6.2f}")
    assert lower <= upper

The lower bound should never exceed the upper bound. If it does, there is a bug in the training code or in the accounting, and finding such bugs is what audits are for. The gap between the two tells you how loose the analysis might be, or how weak the attack is.

### 3. A buggy implementation that the audit catches
A classic mistake: the noise is scaled down by the batch size once too often, so it is 200 times too small. The accountant still reports the same epsilon, but the guarantee is gone.

In [ ]:
def buggy_gd(include_canary, sigma, rng, steps=10, clip=1.0, lr=20.0):
    Xs, Ys = (np.vstack([X, canary_x]), np.vstack([Y, canary_y])) if include_canary else (X, Y)
    W = np.zeros((65, 10))
    for _ in range(steps):
        Z = Xs @ W; P = np.exp(Z - Z.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
        R = P - Ys
        R = R * np.minimum(1, clip / (np.linalg.norm(Xs, axis=1) * np.linalg.norm(R, axis=1) + 1e-12))[:, None]
        G = (Xs.T @ R + sigma * clip * rng.normal(size=W.shape) / len(X)) / len(X)   # BUG: the noise is divided by the batch size twice
        W -= lr * G
    return W[0, 0]

good = noisy_gd
noisy_gd = buggy_gd
tpr, fpr, lower = audit(4.0, trials=6000)
noisy_gd = good
claimed = epsilon(1.0, 4.0, STEPS, DELTA)
print(f"buggy code at sigma 4: TPR {tpr:.3f} FPR {fpr:.3f}  measured lower bound {lower:.2f}  claimed epsilon {claimed:.2f}")
print("VIOLATION: the claim is false" if lower > claimed else "no violation detected with this many trials")
assert lower > claimed

### 4. Auditing in one run, and LLMs
- Training thousands of models is too costly for real systems. Steinke et al. (2023) plant many independent canaries in one run and count how many the attacker gets right.
- For large language models, Duan et al. (2024) found that membership attacks on pretraining data barely beat guessing in most settings. Reasons include huge training sets, few passes over each example, and a blurry line between members and near-duplicates. Weak attacks do not prove privacy: an audit gives a lower bound only.

In [ ]:
# TODO: one-run audit of your session 3 model.
# 1. Make m = 200 canaries (random images with random labels). Include each one with probability 1/2.
# 2. Train once with DP-SGD from session 3.
# 3. Score each canary by its loss. Guess 'in' for the 50 lowest, 'out' for the 50 highest, and abstain on the rest.
# 4. Count correct guesses and use the bound in Steinke et al. (their Theorem 5.2) to turn that count into a lower bound on epsilon.

### TODO
1. Do the one-run audit above.
2. Replace the white-box score with the canary's loss (a black-box score). How much weaker is the lower bound?
3. Why is an outlier a better canary than a typical example?